In [1]:
import os
import sys
from pathlib import Path
from __future__ import annotations


import matplotlib.pyplot as plt
import numpy as np

notebook_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(notebook_dir, ".."))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

from my_nn import (
    MLP,
    accuracy,
    load_mnist,
    train_epoch_sgd,
    build_optimizer_state,
)

In [2]:
def run_training(
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_test: np.ndarray,
    y_test: np.ndarray,
    init_method: str,
    dropout_p: float,
    l2_lambda: float,
    optimizer: str,
    lr: float,
    epochs: int,
    batch_size: int,
    seed: int = 42,
) -> list[float]:
    model = MLP(
        input_dim=784,
        hidden_dims=[96, 64],
        num_classes=10,
        init_method=init_method,
        dropout_p=dropout_p,
        seed=seed,
    )
    opt_state = build_optimizer_state(model, optimizer)
    test_accs: list[float] = []
    for epoch in range(epochs):
        tl, ta = train_epoch_sgd(
            model,
            x_train,
            y_train,
            batch_size=batch_size,
            lr=lr,
            l2_lambda=l2_lambda,
            optimizer=optimizer,
            opt_state=opt_state,
        )
        logits_te = model.forward(x_test, train=False)
        te_acc = accuracy(logits_te, y_test)
        test_accs.append(te_acc)
        if epoch % 5 == 0 or epoch == 1:
            print(
                f"epoch {epoch:3d}  train_loss={tl:.4f}  train_acc={ta:.4f}  test_acc={te_acc:.4f}"
            )
    return test_accs

In [3]:
x_train, y_train, x_test, y_test = load_mnist()
epochs = 35
batch_size = 32

In [ ]:
init_specs = [
    ("Xavier", "xavier", 0.0, 0.0, "sgd", 0.35),
    ("He", "he", 0.0, 0.0, "sgd", 0.35),
    ("小方差高斯", "normal_small", 0.0, 0.0, "sgd", 0.35),
]
curves_init: list[tuple[str, list[float]]] = []
for label, init_m, do, l2, opt, lr in init_specs:
    acc = run_training(
        x_train,
        y_train,
        x_test,
        y_test,
        init_m,
        do,
        l2,
        opt,
        lr,
        epochs,
        batch_size,
        seed=43,
    )
    curves_init.append((label, acc))

epoch   0  train_loss=0.2541  train_acc=0.9588  test_acc=0.9559
epoch   1  train_loss=0.1160  train_acc=0.9703  test_acc=0.9621
epoch   5  train_loss=0.0545  train_acc=0.9785  test_acc=0.9665
epoch  10  train_loss=0.0346  train_acc=0.9833  test_acc=0.9660
epoch  15  train_loss=0.0321  train_acc=0.9872  test_acc=0.9711
epoch  20  train_loss=0.0339  train_acc=0.9917  test_acc=0.9750
epoch  25  train_loss=0.0286  train_acc=0.9908  test_acc=0.9746
epoch  30  train_loss=0.0300  train_acc=0.9942  test_acc=0.9745
epoch   0  train_loss=0.3299  train_acc=0.9424  test_acc=0.9348
epoch   1  train_loss=0.1390  train_acc=0.9533  test_acc=0.9444
epoch   5  train_loss=0.0705  train_acc=0.9776  test_acc=0.9660
epoch  10  train_loss=0.0500  train_acc=0.9827  test_acc=0.9689
epoch  15  train_loss=0.0453  train_acc=0.9858  test_acc=0.9684
epoch  20  train_loss=0.0409  train_acc=0.9860  test_acc=0.9692
